# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library, with all dataset entities referenced via their Croissant `@id` fields. 

### Dataset Source
[View Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"Version: {meta.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their Croissant `@id` fields.

In [ ]:
# List all record sets and their @id
record_sets = list(dataset.record_sets)

print("Available Record Sets and their @id:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# For the main record set, show fields (and their @id)
if record_sets:
    main_rs = record_sets[0]
    print(f"\nFields and fields' @id in Record Set '{main_rs.name}' (@id: {main_rs.id}):")
    for field in main_rs.fields:
        print(f"- {field.name} (@id: {field.id}, dtype: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets by @id
dataframes = {}
rs_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    # Use the @id to reference each record set when extracting records
    df = pd.DataFrame(list(dataset.records(record_set=rs.id)))
    dataframes[rs.id] = df

# Show columns of the first record set DataFrame
main_rs_id = rs_ids[0] if rs_ids else None
if main_rs_id is not None:
    print(f"\nColumns for record set '@id': {main_rs_id}")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing values, and grouping.

Replace field `@id`s below with IDs found in the overview for accurate referencing.

In [ ]:
from pandas.api.types import is_numeric_dtype

# For demonstration: choose a numeric field ID from the first fields listed
main_rs = record_sets[0]
main_rs_id = main_rs.id
numeric_field = None
for field in main_rs.fields:
    if field.data_type in ('Number', 'Float', 'Integer'):
        numeric_field = field.id
        break

if numeric_field is not None:
    df = dataframes[main_rs_id]
    if numeric_field in df.columns and is_numeric_dtype(df[numeric_field]):
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalize the numeric field
        norm_col = numeric_field + '_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical field:
        group_field = None
        for field in main_rs.fields:
            if field.data_type in ('Text', 'String') and field.id != numeric_field:
                if field.id in df.columns:
                    group_field = field.id
                    break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print(f"Field {numeric_field} not found or not numeric in DataFrame columns:", df.columns.tolist())
else:
    print("No numeric field found in fields for main record set.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its grouped means, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    if numeric_field in df.columns and is_numeric_dtype(df[numeric_field]):
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

        # If grouped data is available (from previous grouping)
        if 'grouped_df' in locals() and group_field is not None:
            plt.figure(figsize=(10, 4))
            sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.xticks(rotation=90)
            plt.show()

## 6. Conclusion
- Loaded and explored FAIR² protein dataset via Croissant schema using only `@id` referencing.
- Listed available record sets and their fields (`@id`), and loaded sample data for further analysis.
- Performed filtering, normalization, and grouping based on field IDs as direct references, demonstrating EDA best practices using a standards-based approach.

For further exploration, consult the [FAIR² package metadata](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and use field/record set IDs for all programmatic access with `mlcroissant`.